# Laboratorio: valores y vectores propios

En este laboratorio calcularemos espectros y espacios propios de manera exacta, compararemos multiplicidades, verificaremos propiedades y conectaremos los resultados con la geometría.

## 0. Preparación

In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

sp.init_printing()
np.set_printoptions(precision=6, suppress=True)

## 1. Procedimiento simbólico completo

Usaremos la convención mónica $\chi_A(t)=\det(tI-A)$. Para cada raíz calcularemos el núcleo de $A-\lambda I$, que es el espacio propio correspondiente.

In [ ]:
def estudio_espectral(A):
    A = sp.Matrix(A)
    if A.rows != A.cols:
        raise ValueError("La matriz debe ser cuadrada.")
    t = sp.symbols('t')
    chi = sp.factor(A.charpoly(t).as_expr())
    datos = []
    for lam, mult_alg, base in A.eigenvects():
        M = A - lam * sp.eye(A.rows)
        R, pivotes = M.rref()
        datos.append({
            'valor': lam,
            'multiplicidad_algebraica': mult_alg,
            'multiplicidad_geometrica': len(base),
            'rref': R,
            'base': base,
        })
    return chi, datos

A = sp.Matrix([[2, 1], [1, 2]])
chi_A, datos_A = estudio_espectral(A)
print("Polinomio característico:", chi_A)
for dato in datos_A:
    print("\nValor propio:", dato['valor'])
    print("Multiplicidades algebraica y geométrica:",
          dato['multiplicidad_algebraica'], dato['multiplicidad_geometrica'])
    print("Base del espacio propio:", dato['base'])

In [ ]:
assert chi_A == (sp.symbols('t') - 1) * (sp.symbols('t') - 3)
assert {d['valor'] for d in datos_A} == {sp.Integer(1), sp.Integer(3)}
for d in datos_A:
    for v in d['base']:
        assert A*v == d['valor']*v

## 2. Multiplicidades algebraica y geométrica

La matriz siguiente tiene el valor propio $4$ repetido algebraicamente, pero solo una dirección propia asociada. El valor propio $2$ completa el ejemplo.

In [ ]:
B = sp.Matrix([[4, 1, 0], [0, 4, 0], [0, 0, 2]])
chi_B, datos_B = estudio_espectral(B)
print("Polinomio característico:", chi_B)
for d in datos_B:
    print(f"lambda={d['valor']}: ma={d['multiplicidad_algebraica']}, "
          f"mg={d['multiplicidad_geometrica']}, base={d['base']}")

dato_4 = next(d for d in datos_B if d['valor'] == 4)
assert dato_4['multiplicidad_algebraica'] == 2
assert dato_4['multiplicidad_geometrica'] == 1

### El espacio propio se obtiene mediante eliminación

Para $\lambda=4$, reducimos $B-4I$. El número de variables libres coincide con la multiplicidad geométrica.

In [ ]:
M4 = B - 4*sp.eye(3)
R4, pivotes4 = M4.rref()
print("B - 4I =")
sp.pprint(M4)
print("Forma reducida y columnas pivote:")
sp.pprint(R4)
print(pivotes4)
print("Base de ker(B-4I):", M4.nullspace())
assert len(M4.nullspace()) == 3 - len(pivotes4) == 1

## 3. Una matriz real sin valores propios reales

Consideremos una rotación de $60^\circ$. Su polinomio característico tiene discriminante negativo. SymPy trabaja sobre los complejos y devuelve el par conjugado.

In [ ]:
theta = sp.pi/3
R = sp.Matrix([[sp.cos(theta), -sp.sin(theta)],
               [sp.sin(theta),  sp.cos(theta)]])
t = sp.symbols('t', real=True)
chi_R = sp.factor((t*sp.eye(2) - R).det())
discriminante = sp.discriminant(chi_R, t)
print("Polinomio característico:", chi_R)
print("Discriminante:", discriminante)
print("Valores propios complejos:", R.eigenvals())
assert discriminante < 0
esperados = [sp.cos(theta) + sp.I*sp.sin(theta),
             sp.cos(theta) - sp.I*sp.sin(theta)]
obtenidos = list(R.eigenvals())
assert all(any(sp.simplify(z-w) == 0 for w in esperados) for z in obtenidos)

## 4. Semejanza y cambio de coordenadas

Si $A=PBP^{-1}$, ambas matrices representan el mismo operador en bases distintas. Deben tener el mismo polinomio característico. Si $Av=\lambda v$, entonces $P^{-1}v$ es vector propio de $B$.

In [ ]:
D = sp.diag(1, 3)
P = sp.Matrix([[1, 1], [-1, 1]])
A_sim = sp.simplify(P * D * P.inv())
print("A = P D P^{-1}:")
sp.pprint(A_sim)
assert A_sim.charpoly().as_expr() == D.charpoly().as_expr()

v = P.col(1)
assert A_sim*v == 3*v
assert D*(P.inv()*v) == 3*(P.inv()*v)

## 5. Evaluación de polinomios en una matriz

Si $q(t)=a_0+a_1t+\cdots+a_mt^m$, entonces $q(A)=a_0I+a_1A+\cdots+a_mA^m$. El término constante siempre multiplica a la identidad. Por ejemplo, para $q(t)=-t^3+5t-2$ se tiene $q(A)=-A^3+5A-2I$, no $-A^3+5A-2$. Después verificaremos que, si $Av=\lambda v$, entonces $q(A)v=q(\lambda)v$.

In [ ]:
C = sp.Matrix([[2, 1], [0, 4]])
v2 = sp.Matrix([1, 0])
qC = -C**3 + 5*C - 2*sp.eye(2)
q2 = -2**3 + 5*2 - 2
print("q(2) =", q2)
sp.pprint(qC)
assert C*v2 == 2*v2
assert qC*v2 == q2*v2
assert q2 == 0

## 6. Verificación de Cayley–Hamilton

Para $A=\begin{pmatrix}2&1\\1&2\end{pmatrix}$, el polinomio característico es $t^2-4t+3$. Al sustituir $A$ se obtiene la matriz cero.

In [ ]:
cero_CH = sp.simplify(A**2 - 4*A + 3*sp.eye(2))
sp.pprint(cero_CH)
assert cero_CH == sp.zeros(2)

# Reducción recursiva de A^10 a una combinación de A e I
resto = sp.rem(sp.symbols('t')**10, sp.symbols('t')**2 - 4*sp.symbols('t') + 3,
               domain=sp.QQ)
print("Resto de t^10 módulo chi_A(t):", resto)
A10_reducida = resto.coeff(sp.symbols('t'), 1)*A + resto.coeff(sp.symbols('t'), 0)*sp.eye(2)
assert A**10 == A10_reducida

## 7. Visualización de las direcciones propias

La matriz simétrica $A$ transforma el círculo unitario en una elipse. Las direcciones propias permanecen sobre sus mismas rectas y se escalan por $1$ y $3$.

In [ ]:
A_np = np.array(A, dtype=float)
angulos = np.linspace(0, 2*np.pi, 500)
circulo = np.vstack([np.cos(angulos), np.sin(angulos)])
elipse = A_np @ circulo
u1 = np.array([-1, 1], dtype=float) / np.sqrt(2)
u3 = np.array([1, 1], dtype=float) / np.sqrt(2)

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.plot(circulo[0], circulo[1], '--', color='#888888', label='círculo unitario')
ax.plot(elipse[0], elipse[1], color='#2a6fbb', lw=2.5, label='imagen mediante A')
for u, lam, color in [(u1, 1, '#1b9e77'), (u3, 3, '#d95f02')]:
    ax.quiver(0, 0, u[0], u[1], angles='xy', scale_units='xy', scale=1,
              color=color, width=0.011)
    Au = A_np @ u
    ax.quiver(0, 0, Au[0], Au[1], angles='xy', scale_units='xy', scale=1,
              color=color, width=0.011, alpha=0.65,
              label=fr'dirección propia, $\lambda={lam}$')
ax.axhline(0, color='black', lw=0.6)
ax.axvline(0, color='black', lw=0.6)
ax.set(aspect='equal', xlim=(-3.5, 3.5), ylim=(-3.5, 3.5),
       xlabel='$x_1$', ylabel='$x_2$')
ax.grid(alpha=0.2)
ax.legend(loc='upper left')
plt.show()

## 8. Cálculo numérico y residuos

**numpy.linalg.eig** devuelve los valores propios y una matriz cuyas columnas son vectores propios. Un cálculo numérico debe acompañarse con el residuo $\|Av-\lambda v\|$.

In [ ]:
N = np.array([[0., -1.], [1., 0.]])
valores, vectores = np.linalg.eig(N)
print("Valores propios:", valores)
for j, lam in enumerate(valores):
    v = vectores[:, j]
    residuo = np.linalg.norm(N @ v - lam*v)
    print(f"residuo {j}: {residuo:.3e}")
    assert residuo < 1e-12
assert np.allclose(np.sort_complex(valores), np.array([-1j, 1j]))

## 9. Actividades

1. Aplica **estudio_espectral** a la matriz nilpotente de orden $3$ del texto y explica por qué solo aparece una dirección propia.
2. Construye dos matrices con los mismos valores propios $1$ y $2$ que no sean semejantes entre sí. Pista: considera matrices de orden $3$ y compara multiplicidades geométricas.
3. Verifica simbólicamente las restricciones espectrales de una matriz idempotente, una involutiva y una nilpotente.
4. Cambia la matriz de la visualización por una matriz no simétrica con dos valores propios reales. Compara el ángulo entre las direcciones propias.
5. Para una matriz triangular aleatoria, compara sus entradas diagonales con la salida de **eigenvals()**.

## 10. Cierre

- Los valores propios son raíces del polinomio característico.
- Cada espacio propio se calcula como un núcleo.
- La multiplicidad geométrica nunca supera a la algebraica.
- Las matrices semejantes conservan el espectro.
- En cálculos numéricos conviene verificar siempre los residuos.